# 🔍 Optimización Bayesiana de Hiperparámetros - Estrategia S6

## Federated Proactive Forest - S6 Per-Client Macro-F1

Este notebook ejecuta la optimización bayesiana con Optuna para encontrar los mejores hiperparámetros de la estrategia S6 en el dataset Car.

### Hiperparámetros a optimizar:
- **`f1_weight`**: Peso de F1 en ranking (pcd_weight = 1 - f1_weight) → [0.0, 1.0]
- **`local_weight`**: Peso de modelo local en inferencia híbrida (global_weight = 1 - local_weight) → [0.0, 1.0]
- **`t_max`**: Máximo de árboles en modelo global → [30, 150]

### Métrica objetivo:
- **Macro-F1 global**: Penaliza ignorar clases minoritarias (crítico para Car dataset desbalanceado)

## 1️⃣ Configuración Inicial

In [1]:
# Imports necesarios
import sys
from pathlib import Path

# Añadir project root al path
project_root = Path.cwd().parent.parent.parent.parent
sys.path.insert(0, str(project_root))

import numpy as np
import pandas as pd
import yaml
import optuna
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder, StandardScaler
from sklearn.model_selection import train_test_split

print(f"✅ Project root: {project_root}")
print(f"✅ Optuna version: {optuna.__version__}")

c:\Users\Adrián Rodríguez\AppData\Local\Programs\Python\Python38\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Project root: c:\Users\Adrián Rodríguez\Documents\! Study 📝\📊 KDD 🤖🧠\! Federated learning\federated_proactive_forest
✅ Optuna version: 4.5.0


## 2️⃣ Carga del Dataset Car

In [2]:
# Cargar dataset Car
data_path = project_root / 'data' / 'car.csv'
df = pd.read_csv(data_path)

print(f"📊 Dataset Car: {df.shape[0]} muestras, {df.shape[1]} características")
print(f"\nDistribución Target:")
print(df['class'].value_counts())
print(f"\nColumnas: {df.columns.tolist()}")

📊 Dataset Car: 1728 muestras, 7 características

Distribución Target:
class
unacc    1210
acc       384
good       69
vgood      65
Name: count, dtype: int64

Columnas: ['buying', 'maint', 'doors', 'persons', 'lug_boot', 'safety', 'class']


In [3]:
# Preprocesamiento
target_column = 'class'
feature_columns = [col for col in df.columns if col != target_column]

# Codificar características categóricas
encoder = OrdinalEncoder()
X_encoded = encoder.fit_transform(df[feature_columns])

# Codificar target
le = LabelEncoder()
y_encoded = le.fit_transform(df[target_column])
class_names = list(le.classes_)

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

# Escalar features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print(f"\n✅ Features: {X_train.shape}")
print(f"✅ Train: {X_train.shape[0]} muestras")
print(f"✅ Test: {X_test.shape[0]} muestras")
print(f"✅ Clases: {class_names}")


✅ Features: (1382, 6)
✅ Train: 1382 muestras
✅ Test: 346 muestras
✅ Clases: ['acc', 'good', 'unacc', 'vgood']


In [4]:
# Crear DatasetSplit
from src.domain.dataset.base_adapter import DatasetSplit

dataset_split = DatasetSplit(
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    feature_names=feature_columns,
    class_names=class_names,
    dataset_name='car'
)

print(f"✅ DatasetSplit creado: {dataset_split.dataset_name}")

✅ DatasetSplit creado: car


## 3️⃣ Configuración de Optimización

In [5]:
# Cargar search space desde YAML
search_spaces_path = project_root / 'configs' / 'experiments' / 'optimization' / 'search_spaces.yaml'

with open(search_spaces_path, 'r', encoding='utf-8') as f:
    all_spaces = yaml.safe_load(f)

search_space = all_spaces['S6']

print("🔍 ESPACIO DE BÚSQUEDA PARA S6:")
print("="*60)
for param_name, param_config in search_space.items():
    print(f"  {param_name}:")
    for key, value in param_config.items():
        print(f"    {key}: {value}")
print("="*60)

🔍 ESPACIO DE BÚSQUEDA PARA S6:
  f1_weight:
    type: float
    low: 0.0
    high: 1.0
    step: 0.05
  local_weight:
    type: float
    low: 0.0
    high: 1.0
    step: 0.05
  t_max:
    type: int
    low: 30
    high: 150
    step: 10


In [6]:
# Construir config base
base_config = {
    'federation': {
        'n_clients': 5,
        'distribution': 'iid',
        'seed': 42,
    },
    'model': {
        'n_estimators': 100,  # Fixed por ahora
        'alpha': 0.1,
        'split_criterion': 'entropy',
        'use_progressive_stopping': True,
        'convergence': 0.002,
        'episode_size': 5,
        'verbose': False,
    },
    'aggregation': {
        'strategy': 'S6',
        't_max': 100,
        'f1_weight': 0.5,
        'pcd_weight': 0.5,
        'convergence': 0.002,
        'episode_size': 5,
    },
    'prediction': {
        'local_weight': 0.4,
        'global_weight': 0.6,
    },
    'verbose': False,
    'seed': 42,
}

print("⚙️ CONFIGURACIÓN BASE:")
print("="*60)
print(f"  Estrategia: S6")
print(f"  N_clients: 5")
print(f"  N_estimators: 100 (fijo)")
print(f"  Métrica objetivo: macro_f1")
print("="*60)

⚙️ CONFIGURACIÓN BASE:
  Estrategia: S6
  N_clients: 5
  N_estimators: 100 (fijo)
  Métrica objetivo: macro_f1


## 4️⃣ Ejecutar Optimización

In [7]:
# Importar optimizador
from src.application.hyperparam_optimizer import HyperparamOptimizer

# Crear optimizador
optimizer = HyperparamOptimizer(
    dataset_split=dataset_split,
    strategy='S6',
    base_config=base_config,
    search_space=search_space,
    verbose=True  # Mostrar detalles de cada trial
)

print("🚀 OPTIMIZADOR CREADO EXITOSAMENTE")
print(f"   Estrategia: {optimizer.strategy}")
print(f"   Hiperparámetros a optimizar: {list(search_space.keys())}")

🚀 OPTIMIZADOR CREADO EXITOSAMENTE
   Estrategia: S6
   Hiperparámetros a optimizar: ['f1_weight', 'local_weight', 't_max']


In [8]:
# Ejecutar optimización
N_TRIALS = 10  # Cambiar a 50+ para optimización seria

print(f"\n{'='*60}")
print(f"🚀 INICIANDO OPTIMIZACIÓN: {N_TRIALS} TRIALS")
print(f"{'='*60}\n")

study = optimizer.optimize(
    n_trials=N_TRIALS,
    metric='macro_f1',
    seed=42
)

print(f"\n{'='*60}")
print(f"✅ OPTIMIZACIÓN COMPLETADA")
print(f"{'='*60}")
print(f"Mejor Macro-F1: {study.best_value:.4f}")
print(f"\nMejores hiperparámetros:")
for param, value in study.best_params.items():
    print(f"  {param}: {value}")
print(f"{'='*60}")


🚀 INICIANDO OPTIMIZACIÓN: 10 TRIALS



c:\Users\Adrián Rodríguez\AppData\Local\Programs\Python\Python38\lib\site-packages\optuna\_experimental.py:32: ExperimentalWarning: Argument ``multivariate`` is an experimental feature. The interface can change in the future.
  warnings.warn(
[I 2026-04-03 21:35:45,085] A new study created in memory with name: S6_macro_f1_10trials



🚀 INICIANDO OPTIMIZACIÓN BAYESIANA
   Estrategia: S6
   Métrica: macro_f1
   Trials: 10
   Sampler: TPESampler
   Pruner: MedianPruner



  0%|          | 0/10 [00:00<?, ?it/s]


🔬 TRIAL 1
Params: {'f1_weight': 0.35000000000000003, 'local_weight': 0.9500000000000001, 't_max': 120}
DEBUG _train_local_forests client_0: y_pred_test sample=['good' 'unacc' 'acc' 'unacc' 'acc'] type=<class 'numpy.str_'>
DEBUG _train_local_forests client_0: y_test sample=[1 2 2 2 0] type=<class 'numpy.int32'>
DEBUG _train_local_forests client_1: y_pred_test sample=['acc' 'unacc' 'unacc' 'unacc' 'acc'] type=<class 'numpy.str_'>
DEBUG _train_local_forests client_1: y_test sample=[1 2 2 2 0] type=<class 'numpy.int32'>
DEBUG _train_local_forests client_2: y_pred_test sample=['good' 'unacc' 'acc' 'unacc' 'acc'] type=<class 'numpy.str_'>
DEBUG _train_local_forests client_2: y_test sample=[1 2 2 2 0] type=<class 'numpy.int32'>
DEBUG _train_local_forests client_3: y_pred_test sample=['acc' 'unacc' 'unacc' 'unacc' 'acc'] type=<class 'numpy.str_'>
DEBUG _train_local_forests client_3: y_test sample=[1 2 2 2 0] type=<class 'numpy.int32'>
DEBUG _train_local_forests client_4: y_pred_test sample=['

...

Best trial: 8. Best value: 0.818303:  90%|█████████ | 9/10 [01:06<00:08,  8.35s/it]


🔬 TRIAL 10
Params: {'f1_weight': 0.5, 'local_weight': 0.6000000000000001, 't_max': 30}
DEBUG _train_local_forests client_0: y_pred_test sample=['good' 'unacc' 'acc' 'unacc' 'acc'] type=<class 'numpy.str_'>
DEBUG _train_local_forests client_0: y_test sample=[1 2 2 2 0] type=<class 'numpy.int32'>
DEBUG _train_local_forests client_1: y_pred_test sample=['acc' 'unacc' 'acc' 'unacc' 'acc'] type=<class 'numpy.str_'>
DEBUG _train_local_forests client_1: y_test sample=[1 2 2 2 0] type=<class 'numpy.int32'>
DEBUG _train_local_forests client_2: y_pred_test sample=['good' 'unacc' 'acc' 'unacc' 'acc'] type=<class 'numpy.str_'>
DEBUG _train_local_forests client_2: y_test sample=[1 2 2 2 0] type=<class 'numpy.int32'>
DEBUG _train_local_forests client_3: y_pred_test sample=['acc' 'unacc' 'unacc' 'unacc' 'acc'] type=<class 'numpy.str_'>
DEBUG _train_local_forests client_3: y_test sample=[1 2 2 2 0] type=<class 'numpy.int32'>
DEBUG _train_local_forests client_4: y_pred_test sample=['acc' 'unacc' 'unac

[I 2026-04-03 21:36:56,528] Trial 9 pruned. 


Best trial: 8. Best value: 0.818303: 100%|██████████| 10/10 [01:11<00:00,  7.14s/it]



✅ OPTIMIZACIÓN COMPLETADA
   Mejor trial: 8
   Mejor macro_f1: 0.8183

   Mejores hiperparámetros:
      f1_weight: 0.45
      local_weight: 0.8
      t_max: 50


✅ OPTIMIZACIÓN COMPLETADA
Mejor Macro-F1: 0.8183

Mejores hiperparámetros:
  f1_weight: 0.45
  local_weight: 0.8
  t_max: 50


## 5️⃣ Guardar Resultados

In [9]:
# Guardar resultados
from src.infrastructure.persistence.results_logger import OptimizationResultsLogger

output_dir = project_root / 'results' / 'opt_s6_car_notebook'
logger = OptimizationResultsLogger(str(output_dir))

metadata = {
    'strategy': 'S6',
    'dataset': 'car',
    'metric': 'macro_f1',
    'n_trials': N_TRIALS,
    'n_clients': 5,
    'n_estimators': 100,
    'seed': 42,
}

saved_files = logger.save_study(study, metadata=metadata)

print(f"\n💾 RESULTADOS GUARDADOS:")
for file_type, file_path in saved_files.items():
    print(f"  {file_type}: {file_path}")

📄 Trials guardados a: c:\Users\Adrián Rodríguez\Documents\! Study 📝\📊 KDD 🤖🧠\! Federated learning\federated_proactive_forest\results\opt_s6_car_notebook\trials.csv
📋 Best config guardado a: c:\Users\Adrián Rodríguez\Documents\! Study 📝\📊 KDD 🤖🧠\! Federated learning\federated_proactive_forest\results\opt_s6_car_notebook\best_config.yaml


c:\Users\Adrián Rodríguez\Documents\! Study 📝\📊 KDD 🤖🧠\! Federated learning\federated_proactive_forest\src\infrastructure\persistence\results_logger.py:79: UserWarning: the 'timedelta' type is not supported, and will be written as integer values (ns frequency) to the database.
  df_trials.to_sql('trials', conn, if_exists='replace', index=False)


🗄️  Base de datos guardada a: c:\Users\Adrián Rodríguez\Documents\! Study 📝\📊 KDD 🤖🧠\! Federated learning\federated_proactive_forest\results\opt_s6_car_notebook\optimization.db
📦 Study object guardado a: c:\Users\Adrián Rodríguez\Documents\! Study 📝\📊 KDD 🤖🧠\! Federated learning\federated_proactive_forest\results\opt_s6_car_notebook\study.pkl

💾 RESULTADOS GUARDADOS:
  csv: c:\Users\Adrián Rodríguez\Documents\! Study 📝\📊 KDD 🤖🧠\! Federated learning\federated_proactive_forest\results\opt_s6_car_notebook\trials.csv
  yaml: c:\Users\Adrián Rodríguez\Documents\! Study 📝\📊 KDD 🤖🧠\! Federated learning\federated_proactive_forest\results\opt_s6_car_notebook\best_config.yaml
  sqlite: c:\Users\Adrián Rodríguez\Documents\! Study 📝\📊 KDD 🤖🧠\! Federated learning\federated_proactive_forest\results\opt_s6_car_notebook\optimization.db
  study_pickle: c:\Users\Adrián Rodríguez\Documents\! Study 📝\📊 KDD 🤖🧠\! Federated learning\federated_proactive_forest\results\opt_s6_car_notebook\study.pkl


## 6️⃣ Visualizaciones

In [10]:
# Generar plots
plot_files = logger.generate_plots(study)

print(f"\n📊 VISUALIZACIONES GENERADAS:")
for plot_type, plot_path in plot_files.items():
    print(f"  {plot_type}: {plot_path}")

📈 Optimization history: c:\Users\Adrián Rodríguez\Documents\! Study 📝\📊 KDD 🤖🧠\! Federated learning\federated_proactive_forest\results\opt_s6_car_notebook\optimization_history.html
📊 Parameter importance: c:\Users\Adrián Rodríguez\Documents\! Study 📝\📊 KDD 🤖🧠\! Federated learning\federated_proactive_forest\results\opt_s6_car_notebook\param_importance.html
🔀 Parallel coordinate: c:\Users\Adrián Rodríguez\Documents\! Study 📝\📊 KDD 🤖🧠\! Federated learning\federated_proactive_forest\results\opt_s6_car_notebook\parallel_coordinate.html
🗺️  Contour plot: c:\Users\Adrián Rodríguez\Documents\! Study 📝\📊 KDD 🤖🧠\! Federated learning\federated_proactive_forest\results\opt_s6_car_notebook\contour.html
📉 Slice plot: c:\Users\Adrián Rodríguez\Documents\! Study 📝\📊 KDD 🤖🧠\! Federated learning\federated_proactive_forest\results\opt_s6_car_notebook\slice.html
⏱️  Timeline: c:\Users\Adrián Rodríguez\Documents\! Study 📝\📊 KDD 🤖🧠\! Federated learning\federated_proactive_forest\results\opt_s6_car_notebook\

In [11]:
# Mostrar optimization history directamente en notebook
fig = optuna.visualization.plot_optimization_history(study)
fig.update_layout(
    title="S6 Optimization History - Car Dataset",
    width=800,
    height=500
)
fig.show()

In [12]:
# Mostrar parameter importance
fig = optuna.visualization.plot_param_importances(study)
fig.update_layout(
    title="S6 Parameter Importance - Car Dataset",
    width=800,
    height=500
)
fig.show()

In [13]:
# Mostrar parallel coordinate plot
fig = optuna.visualization.plot_parallel_coordinate(study)
fig.update_layout(
    title="S6 Parallel Coordinate - Car Dataset",
    width=900,
    height=600
)
fig.show()

## 7️⃣ Reporte Resumen

In [14]:
# Generar reporte resumen
report_path = logger.generate_summary_report(study, metadata=metadata)
print(f"\n📝 Reporte guardado en: {report_path}")


📝 Summary report: c:\Users\Adrián Rodríguez\Documents\! Study 📝\📊 KDD 🤖🧠\! Federated learning\federated_proactive_forest\results\opt_s6_car_notebook\summary_report.txt

OPTIMIZATION SUMMARY REPORT

Strategy: S6
Dataset: car
Metric: macro_f1

----------------------------------------------------------------------
RESULTS SUMMARY
----------------------------------------------------------------------
Total trials: 10
Completed trials: 8
Pruned trials: 2
Failed trials: 0

----------------------------------------------------------------------
BEST RESULT
----------------------------------------------------------------------
Best trial: #8
Best macro_f1: 0.818303

Best hyperparameters:
  - f1_weight: 0.45
  - local_weight: 0.8
  - t_max: 50

----------------------------------------------------------------------
STATISTICS (completed trials)
----------------------------------------------------------------------
Mean macro_f1: 0.787789
Std macro_f1: 0.022836
Min macro_f1: 0.751695
Max macro_f1

## 8️⃣ Análisis de Resultados

In [15]:
# Analizar resultados
df_trials = study.trials_dataframe()
completed_trials = df_trials[df_trials['state'] == 'COMPLETE']

print("\n" + "="*60)
print("📈 ANÁLISIS DE RESULTADOS")
print("="*60)
print(f"Total trials: {len(study.trials)}")
print(f"Completados: {len(completed_trials)}")
print(f"Pruned: {len(df_trials[df_trials['state'] == 'PRUNED'])}")
print(f"Failed: {len(df_trials[df_trials['state'] == 'FAIL'])}")

if len(completed_trials) > 0:
    print(f"\nMejor Macro-F1: {completed_trials['value'].max():.4f}")
    print(f"Peor Macro-F1: {completed_trials['value'].min():.4f}")
    print(f"Media Macro-F1: {completed_trials['value'].mean():.4f}")
    print(f"Std Macro-F1: {completed_trials['value'].std():.4f}")

print("="*60)


📈 ANÁLISIS DE RESULTADOS
Total trials: 10
Completados: 8
Pruned: 2
Failed: 0

Mejor Macro-F1: 0.8183
Peor Macro-F1: 0.7517
Media Macro-F1: 0.7878
Std Macro-F1: 0.0228


In [16]:
# Top 5 trials
print("\n🏆 TOP 5 TRIALS:")
print("="*60)
top5 = completed_trials.nlargest(5, 'value')
for _, row in top5.iterrows():
    print(f"\nTrial #{row['number']}: Macro-F1 = {row['value']:.4f}")
    for col in df_trials.columns:
        if col.startswith('params_'):
            param_name = col.replace('params_', '')
            print(f"  {param_name}: {row[col]}")
print("="*60)


🏆 TOP 5 TRIALS:

Trial #8: Macro-F1 = 0.8183
  f1_weight: 0.45
  local_weight: 0.8
  t_max: 50

Trial #4: Macro-F1 = 0.8052
  f1_weight: 0.8500000000000001
  local_weight: 0.2
  t_max: 50

Trial #7: Macro-F1 = 0.8017
  f1_weight: 0.1
  local_weight: 0.30000000000000004
  t_max: 70

Trial #5: Macro-F1 = 0.7949
  f1_weight: 0.15000000000000002
  local_weight: 0.30000000000000004
  t_max: 90

Trial #1: Macro-F1 = 0.7894
  f1_weight: 0.6000000000000001
  local_weight: 0.15000000000000002
  t_max: 50


## 9️⃣ Comparación con Baseline

In [17]:
# Comparar con baseline (f1_weight=0.5, local_weight=0.4, t_max=100)
baseline_params = {'f1_weight': 0.5, 'local_weight': 0.4, 't_max': 100}
baseline_f1 = None  # Se obtendría ejecutando con baseline

print("\n" + "="*60)
print("📊 COMPARACIÓN CON BASELINE")
print("="*60)
print(f"Baseline (f1_weight=0.5, local_weight=0.4, t_max=100): TBD")
print(f"Optimizado: {study.best_value:.4f}")
print(f"\nMejores parámetros encontrados:")
for param, value in study.best_params.items():
    baseline_val = baseline_params.get(param, 'N/A')
    print(f"  {param}: {baseline_val} → {value}")
print("="*60)


📊 COMPARACIÓN CON BASELINE
Baseline (f1_weight=0.5, local_weight=0.4, t_max=100): TBD
Optimizado: 0.8183

Mejores parámetros encontrados:
  f1_weight: 0.5 → 0.45
  local_weight: 0.4 → 0.8
  t_max: 100 → 50


## ✅ Conclusiones

### Resumen de la optimización:
1. **Hiperparámetros optimizados**: f1_weight, local_weight, t_max
2. **Mejor Macro-F1 alcanzada**: {study.best_value:.4f}
3. **Mejor configuración**: {study.best_params}

### Próximos pasos:
1. Ejecutar con más trials (50-100) para mejor exploración
2. Validar en otros datasets (Letter, Students)
3. Comparar con otras estrategias (S4, S7, PW)
4. Optimizar también n_estimators en el futuro